In [3]:
import torch

print("PyTorch版本：", torch.__version__)
print("CUDA是否可用：", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU型号：", torch.cuda.get_device_name(0))
    print("GPU数量：", torch.cuda.device_count())
    print("CUDA版本：", torch.version.cuda)


from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "/root/autodl-tmp/models/models/Qwen--Qwen2.5-7B-Instruct/snapshots/master"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.float16,
    local_files_only=True
).to("cuda")

model.eval()

print("模型和 tokenizer 加载完成")
print("模型设备：", next(model.parameters()).device)




PyTorch版本： 2.8.0+cu128
CUDA是否可用： True
GPU型号： NVIDIA GeForce RTX 4090
GPU数量： 1
CUDA版本： 12.8


Loading weights:   0%|          | 0/339 [00:01<?, ?it/s]

模型和 tokenizer 加载完成
模型设备： cuda:0


In [1]:
import os
import re
import json
import torch
from datetime import datetime
from docx import Document
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed
)

# =========================================================
# 1. 基础配置
# =========================================================

model_path = "/root/autodl-tmp/models/models/Qwen--Qwen2.5-7B-Instruct/snapshots/master"

# 案例档案
archive_path = "/root/北京市人民政府对《市城市规划设计研究院关于金海风景区总体规划请示》的批复.docx"

# 三份规范文件：修改成你在 AutoDL 中的实际路径
law_path = "/root/中华人民共和国档案法.docx"
regulation_path = "/root/中华人民共和国档案法实施条例.docx"
open_rules_path = "/root/国家档案馆档案开放办法.docx"

output_dir = "/root/autodl-tmp/archive_review_output"
os.makedirs(output_dir, exist_ok=True)

set_seed(42)


# =========================================================
# 2. 加载 Qwen2.5-7B-Instruct
#    使用8-bit，避免24GB 4090显存不足
# =========================================================

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True
)

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True
)

model.eval()

print("模型和 tokenizer 加载完成")
print("模型加载方式：8-bit")
print("计算类型：float16")
print("模型设备：", model.device)


# =========================================================
# 3. 读取档案，并给每个段落增加唯一编号
# =========================================================

def read_archive_with_ids(path):
    doc = Document(path)

    paragraphs = []
    paragraph_dict = {}

    idx = 1

    for p in doc.paragraphs:
        text = p.text.strip()

        if not text:
            continue

        pid = f"D{idx:03d}"

        paragraphs.append(
            f"[{pid}] {text}"
        )

        paragraph_dict[pid] = text

        idx += 1

    return "\n".join(paragraphs), paragraph_dict


archive_text, archive_paragraphs = read_archive_with_ids(
    archive_path
)

print("\n档案读取完成")
print("有效段落数：", len(archive_paragraphs))
print("正文字符数：", len(archive_text))


# =========================================================
# 4. 从法规Word中解析“第X条”
# =========================================================

def read_docx_plain(path):
    doc = Document(path)

    texts = []

    for p in doc.paragraphs:
        text = p.text.strip()

        if text:
            texts.append(text)

    return "\n".join(texts)


def chinese_to_int(chinese):
    """
    这里只处理1—99，足够当前三份规范使用。
    """

    nums = {
        "一": 1,
        "二": 2,
        "三": 3,
        "四": 4,
        "五": 5,
        "六": 6,
        "七": 7,
        "八": 8,
        "九": 9
    }

    if chinese == "十":
        return 10

    if "十" in chinese:

        left, right = chinese.split("十")

        if left == "":
            tens = 1
        else:
            tens = nums.get(left, 0)

        if right == "":
            units = 0
        else:
            units = nums.get(right, 0)

        return tens * 10 + units

    return nums.get(chinese, None)


def parse_articles(path):
    """
    按“段落起始位置”识别法规条文，避免把条文正文中的
    “根据本办法第八条”“对于《档案法》第二十七条”等
    交叉引用误识别为新的条文边界。

    返回：
    {
        27: "第二十七条 ……",
        28: "第二十八条 ……"
    }
    """

    doc = Document(path)

    # 只匹配“段落开头”的条文标题；不匹配正文内部的交叉引用
    article_start = re.compile(
        r"^第([一二三四五六七八九十百]+)条"
    )

    # 章节标题不属于任何条文正文，遇到时跳过
    chapter_heading = re.compile(
        r"^第[一二三四五六七八九十百]+章"
    )

    articles = {}
    current_number = None
    current_parts = []

    def flush_current():
        if current_number is not None and current_parts:
            articles[current_number] = "\\n".join(current_parts).strip()

    for p in doc.paragraphs:
        text = p.text.strip()
        if not text:
            continue

        # 跳过章节标题，避免“第四章 开放档案利用和保护”
        # 被错误拼入上一条条文
        if chapter_heading.match(text):
            continue

        m = article_start.match(text)

        if m:
            flush_current()
            current_number = chinese_to_int(m.group(1))
            current_parts = [text]
        elif current_number is not None:
            current_parts.append(text)

    flush_current()

    return articles


def validate_rule_package(rule_dict):
    """发布前的规则包完整性检查。"""
    problems = []

    for rid, info in rule_dict.items():
        expected = info["article"]
        text = info["text"].strip()

        # 最低长度检查：可快速发现“第三十一条 对于《档案法》”这类截断
        if len(text) < 25:
            problems.append(f"{rid}: 条文疑似过短（{len(text)} chars）")

        # 条文起始编号检查
        if not re.match(r"^第[一二三四五六七八九十百]+条", text):
            problems.append(f"{rid}: 未以完整条文标题开头")

    if problems:
        raise ValueError("规则包完整性检查失败：\\n" + "\\n".join(problems))

    return True


law_articles = parse_articles(law_path)
regulation_articles = parse_articles(regulation_path)
open_articles = parse_articles(open_rules_path)

print("\n法规解析完成")
print("《档案法》条数：", len(law_articles))
print("《档案法》已解析条号：", sorted(law_articles.keys()))
print("《实施条例》条数：", len(regulation_articles))
print("《实施条例》已解析条号：", sorted(regulation_articles.keys()))
print("《开放办法》条数：", len(open_articles))
print("《开放办法》已解析条号：", sorted(open_articles.keys()))


# =========================================================
# 5. 只调取本次开放审核真正需要的条款
#
# 不再把三部法律法规全文塞给模型。
# =========================================================

selected_rules = {

    # 《档案法》
    "R01-A20": {
        "source": "《中华人民共和国档案法》",
        "article": 20,
        "text": law_articles[20]
    },

    "R01-A27": {
        "source": "《中华人民共和国档案法》",
        "article": 27,
        "text": law_articles[27]
    },

    "R01-A28": {
        "source": "《中华人民共和国档案法》",
        "article": 28,
        "text": law_articles[28]
    },

    "R01-A30": {
        "source": "《中华人民共和国档案法》",
        "article": 30,
        "text": law_articles[30]
    },

    "R01-A32": {
        "source": "《中华人民共和国档案法》",
        "article": 32,
        "text": law_articles[32]
    },

    # 《档案法实施条例》
    "R02-A30": {
        "source": "《中华人民共和国档案法实施条例》",
        "article": 30,
        "text": regulation_articles[30]
    },

    "R02-A31": {
        "source": "《中华人民共和国档案法实施条例》",
        "article": 31,
        "text": regulation_articles[31]
    },

    # 《国家档案馆档案开放办法》
    "R03-A07": {
        "source": "《国家档案馆档案开放办法》",
        "article": 7,
        "text": open_articles[7]
    },

    "R03-A08": {
        "source": "《国家档案馆档案开放办法》",
        "article": 8,
        "text": open_articles[8]
    },

    "R03-A09": {
        "source": "《国家档案馆档案开放办法》",
        "article": 9,
        "text": open_articles[9]
    },

    "R03-A12": {
        "source": "《国家档案馆档案开放办法》",
        "article": 12,
        "text": open_articles[12]
    },

    "R03-A13": {
        "source": "《国家档案馆档案开放办法》",
        "article": 13,
        "text": open_articles[13]
    },

    "R03-A14": {
        "source": "《国家档案馆档案开放办法》",
        "article": 14,
        "text": open_articles[14]
    },

    "R03-A15": {
        "source": "《国家档案馆档案开放办法》",
        "article": 15,
        "text": open_articles[15]
    },

    "R03-A17": {
        "source": "《国家档案馆档案开放办法》",
        "article": 17,
        "text": open_articles[17]
    }
}


# 发布/实验前先核验规则包，发现截断则直接停止运行
validate_rule_package(selected_rules)
print("规则包完整性检查通过")


# =========================================================
# 6. 将规则包转换成模型可读文本
# =========================================================

def build_rule_text(rule_dict):

    blocks = []

    for rid, info in rule_dict.items():

        block = (
            f"[{rid}]\n"
            f"规范名称：{info['source']}\n"
            f"条款原文：{info['text']}"
        )

        blocks.append(block)

    return "\n\n".join(blocks)


rules_text = build_rule_text(selected_rules)

print("\n本次实际调用规则数量：", len(selected_rules))


# =========================================================
# 7. 案例元数据
# =========================================================

case_metadata = """
题名：北京市人民政府对《市城市规划设计研究院关于金海风景区总体规划请示》的批复
文号：京政发〔1992〕3号
形成主体：北京市人民政府
形成时间：1992年1月16日
所附文件：北京市城市规划设计研究院《关于金海风景区总体规划的请示》
所附文件形成时间：1991年10月4日
关联地域：北京市平谷县金海地区

审核时点：2026年
形成是否已满25年：是

历史密级：未提供
解密状态：未提供
政府信息公开情况：本次模型输入不提供
现实开放结果：本次模型输入不提供
"""


# =========================================================
# 8. System Prompt
# =========================================================

system_prompt = """
你是一个档案开放审核辅助模型。

你的职责是根据本次明确提供的：
1. 档案对象信息；
2. 档案完整正文；
3. 开放审核规则包；

形成供档案工作人员复核的辅助审核意见。

你不是最终审核责任主体。

你必须严格执行以下原则：

第一，事实来源约束。
所有事实只能来自带有[Dxxx]编号的档案正文。
不得补充正文中不存在的事实。
事实证据匹配约束。
引用的[Dxxx]段落必须能够直接支持对应事实判断，不得仅引用存在但与判断无关的段落。
对“形成时间”“文号”“形成主体”等明确事实，应优先引用实际包含该信息的具体段落。
例如，判断档案形成时间时，必须引用包含日期的段落，不得仅引用题名、受文单位、署名等无关段落。

第二，规范来源约束。
只能使用本次提供的[Rxx-Axx]规则。
不得调用记忆中的法律条文，不得自行补充其他规范。

第三，规范原文约束。
每次适用规范时，必须：
1. 写出规则编号；
2. 引用规则包中能够支持判断的原句；
3. 再说明该原句与档案事实之间的关系。

不得将档案中的事实内容写进法律规范，
不得将自己的解释伪装成法律条文内容。

第四，风险证据约束。
只有档案中的具体事实与规则包规定的限制开放条件
形成明确对应关系时，才能认定存在相应开放风险。

“规划”“道路”“建筑”“投资”“给排水”
“环境保护”“基础设施”等一般行政管理内容，
本身不能直接作为国家秘密、国家安全、
重大利益或第三方权益风险的依据。

否定性风险判断约束。
当判断“未发现具体限制开放事实”时，应说明该结论基于对本次提供档案正文的整体审核，而不能仅以少数无关段落作为依据。
如输出档案依据，可使用能够体现全文审核范围的段落区间，例如[D001]-[D048]，或明确表述“经全文逐段检查”。

风险—规范对应约束。
对国家秘密、国家安全或重大利益、知识产权或个人信息、其他依法应限制利用风险，应优先依据[R03-A08]中对应的四类延期开放情形逐项判断。
如另有更具体规范，可作为辅助依据，但不得用与该风险类型不直接对应的条款替代[R03-A08]的对应项。

第五，未知信息约束。
“历史密级未提供”“解密状态未提供”
表示现有材料不足。

必须在最终输出的“需人工核验事项”中逐项列明，
不得写成“无”，也不得据此直接推定存在国家秘密风险。
这些未知信息本身不等于限制开放事实，但必须交由人工复核。

第六，开放审核逻辑。
对于形成已满25年的档案，
先依据开放期限规则判断是否进入到期开放审核，
再逐项检查是否具有规则包明确规定的限制开放情形。

没有识别到具体限制开放事实时，
不得仅以“可能存在风险”为理由建议限制开放。

第七，输出只是辅助意见。
不得声称作出最终行政决定。
"""


# =========================================================
# 9. User Prompt
# =========================================================

user_prompt = f"""
请对以下真实历史政府文件进行档案开放审核辅助分析。

============================================================
一、审核对象信息
============================================================

{case_metadata}


============================================================
二、档案正文
============================================================

正文已经按段落赋予唯一编号[D001]、[D002]……

引用档案事实时必须给出相应段落编号。

{archive_text}


============================================================
三、本次开放审核规则包
============================================================

以下规则已经由审核系统从规范库中调取。

只能使用这些规则进行规范分析：

{rules_text}


============================================================
四、审核步骤
============================================================

必须依次完成以下三个理由层次。


【第一层：事实识别理由】

不要罗列档案全部内容。

只识别与“是否可以向社会开放”真正有关的事实。

重点回答：

1. 档案形成时间是否已达到法定开放年限；
2. 正文中是否存在能够具体对应限制开放规则的内容；
3. 是否存在无法由正文确认、需要人工进一步核验的信息。

每项判断必须给出[Dxxx]段落编号。


【第二层：规范适用理由】

逐项将第一层识别的事实与规则包进行匹配。

每项规范判断必须包含：

- 对应档案事实；
- 档案段落编号；
- 规则编号；
- 规则原文中的直接依据；
- 适用分析；
- 判断状态。

判断状态只能为：

“明确适用”
“未发现适用事实”
“现有材料不足，需人工核验”

特别注意：

不能把“基础设施建设、道路规划、建筑规划、
环境保护”等档案事实改写成法律条文的内容。


【第三层：风险衡量理由】

在前两层基础上进行综合判断。

分别判断：

1. 国家秘密风险；
2. 国家安全或重大利益风险；
3. 知识产权或个人信息风险；
4. 其他依法应限制利用的风险。

每一种风险都必须说明：

- 是否发现具体事实；
- 事实段落编号；
- 规范依据；
- 判断理由。

如果没有发现具体事实，
明确写“未发现现有材料能够支持该风险的具体事实”。

不能使用“可能涉及敏感信息”
这种没有事实和规范依据的笼统表达。


============================================================
五、模型辅助处理建议
============================================================

处理建议只能从以下三项中选择一项：

A. 建议开放
B. 建议延期开放
C. 现有材料不足，需补充核验后判断

选择后必须说明：

1. 档案是否已达到法定开放年限；
2. 是否发现具体限制开放事实；
3. 哪些信息仍需人工核验；
4. 为什么当前证据支持这一建议。


============================================================
六、严格输出格式
============================================================

一、事实识别理由

事实1
- 事实内容：
- 档案依据：
- 与开放审核的关系：
- 判断：

事实2
……

二、规范适用理由

规范判断1
- 对应事实：
- 档案依据：
- 规则编号：
- 规则原文：必须摘录规则包中直接支持本判断的原句，不得只写“第二十七条”“第七条”等条号
- 适用分析：
- 判断状态：

规范判断2
……

三、风险衡量理由

1. 国家秘密风险
- 相关事实：
- 档案依据：
- 规范依据：
- 判断：

2. 国家安全或重大利益风险
- 相关事实：
- 档案依据：
- 规范依据：
- 判断：

3. 知识产权或个人信息风险
- 相关事实：
- 档案依据：
- 规范依据：
- 判断：

4. 其他限制开放风险
- 相关事实：
- 档案依据：
- 规范依据：
- 判断：

四、模型辅助处理建议

- 建议：
- 开放年限判断：
- 限制开放事实判断：
- 需人工核验事项：
- 综合理由：
"""


# =========================================================
# 10. 构造Chat Template
# =========================================================

messages = [
    {
        "role": "system",
        "content": system_prompt
    },
    {
        "role": "user",
        "content": user_prompt
    }
]

formatted_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer(
    formatted_text,
    return_tensors="pt",
    truncation=False
)

model_inputs = {
    k: v.to(model.device)
    for k, v in model_inputs.items()
}

input_token_count = model_inputs["input_ids"].shape[1]

print("\n模型输入构造完成")
print("输入 token 数：", input_token_count)

# 防止输入意外超过模型实验设计长度
if input_token_count > 12000:
    raise ValueError(
        f"输入过长：{input_token_count} tokens。"
        "请检查是否错误地将完整法规全文再次放入Prompt。"
    )


# =========================================================
# 11. 生成
#
# 对论文原型建议不用sampling，保证可重复
# =========================================================

with torch.no_grad():

    generated_ids = model.generate(
        **model_inputs,

        max_new_tokens=1800,

        # 关闭随机采样，提高可复现性
        do_sample=False,

        repetition_penalty=1.03,

        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )


# 去除输入部分
output_ids = generated_ids[0][
    model_inputs["input_ids"].shape[1]:
]

response = tokenizer.decode(
    output_ids,
    skip_special_tokens=True
)


print("\n" + "=" * 80)
print("Qwen2.5-7B-Instruct 档案开放审核辅助意见")
print("=" * 80)

print(response)


# =========================================================
# 12. 对模型使用的规则编号进行自动校验
#
# 用于发现模型引用不存在的规则
# =========================================================

def validate_rule_ids(response_text, rule_dict):

    used_ids = sorted(
        set(
            re.findall(
                r"R\d{2}-A\d{2}",
                response_text
            )
        )
    )

    invalid_ids = [
        rid
        for rid in used_ids
        if rid not in rule_dict
    ]

    return used_ids, invalid_ids


used_rule_ids, invalid_rule_ids = validate_rule_ids(
    response,
    selected_rules
)

print("\n模型使用规则：", used_rule_ids)

if invalid_rule_ids:
    print("警告：发现未提供的规则编号：", invalid_rule_ids)
else:
    print("规则编号校验通过：未发现未提供规则。")


# =========================================================
# 13. 对档案段落编号进行自动校验
# =========================================================

def validate_paragraph_ids(response_text, paragraph_dict):

    used_ids = sorted(
        set(
            re.findall(
                r"D\d{3}",
                response_text
            )
        )
    )

    invalid_ids = [
        pid
        for pid in used_ids
        if pid not in paragraph_dict
    ]

    return used_ids, invalid_ids


used_doc_ids, invalid_doc_ids = validate_paragraph_ids(
    response,
    archive_paragraphs
)

print("\n模型引用档案段落：", used_doc_ids)

if invalid_doc_ids:
    print("警告：发现不存在的档案段落编号：", invalid_doc_ids)
else:
    print("档案段落编号校验通过。")


# =========================================================
# 14. 输出约束合规性检查
#
# 不修改模型原始输出，只标记是否违反预设约束。
# =========================================================

def validate_output_compliance(response_text, case_metadata_text):
    issues = []

    missing_classification = "历史密级：未提供" in case_metadata_text
    missing_declassification = "解密状态：未提供" in case_metadata_text

    if (missing_classification or missing_declassification) and re.search(
        r"需人工核验事项[：:]\\s*(无|无。)", response_text
    ):
        issues.append(
            "案例存在未提供的历史密级/解密状态，但模型将需人工核验事项写为‘无’。"
        )

    # 检查“规则原文”是否疑似只写条号，而未摘录支持性原句
    for line in response_text.splitlines():
        if "规则原文" in line:
            value = line.split("：", 1)[-1].strip() if "：" in line else ""
            if value and len(value) < 18:
                issues.append(
                    f"规则原文引用疑似过短，需人工核验：{line.strip()}"
                )

    return issues


compliance_issues = validate_output_compliance(
    response,
    case_metadata
)

if compliance_issues:
    print("\\n输出约束合规性检查：发现问题")
    for item in compliance_issues:
        print("-", item)
else:
    print("\\n输出约束合规性检查通过")


# =========================================================
# 15. 保存“实际调用的规则包”
# =========================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

rule_package_path = os.path.join(
    output_dir,
    f"rule_package_{timestamp}.json"
)

with open(
    rule_package_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        selected_rules,
        f,
        ensure_ascii=False,
        indent=2
    )


# =========================================================
# 16. 保存模型原始输出
# =========================================================

raw_output_path = os.path.join(
    output_dir,
    f"archive_review_raw_{timestamp}.txt"
)

with open(
    raw_output_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(response)


# =========================================================
# 17. 保存本次完整审核运行记录
# =========================================================

record = {

    "task_id": f"ARCHIVE_REVIEW_{timestamp}",

    "task_type": "档案开放审核辅助原型",

    "case": {
        "title": "北京市人民政府对《市城市规划设计研究院关于金海风景区总体规划请示》的批复",
        "document_no": "京政发〔1992〕3号",
        "formation_date": "1992-01-16",
        "formation_unit": "北京市人民政府",
        "historical_classification": "未提供",
        "declassification_status": "未提供"
    },

    "model": {
        "name": "Qwen2.5-7B-Instruct",
        "path": model_path,
        "quantization": "8-bit",
        "device": str(model.device)
    },

    "generation_config": {
        "max_new_tokens": 1800,
        "do_sample": False,
        "repetition_penalty": 1.03,
        "seed": 42
    },

    "input": {
        "input_token_count": int(input_token_count),

        # 实际档案输入
        "archive_paragraph_count": len(archive_paragraphs),

        # 实际调用的法规
        "rule_ids": list(selected_rules.keys()),

        # 保存prompt版本
        "system_prompt": system_prompt,
        "user_prompt": user_prompt
    },

    "model_output": {
        "raw_text": response,

        "used_rule_ids": used_rule_ids,
        "invalid_rule_ids": invalid_rule_ids,

        "used_archive_paragraph_ids": used_doc_ids,
        "invalid_archive_paragraph_ids": invalid_doc_ids,
        "compliance_issues": compliance_issues
    },

    "run_time": timestamp
}


record_path = os.path.join(
    output_dir,
    f"archive_review_record_{timestamp}.json"
)

with open(
    record_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        record,
        f,
        ensure_ascii=False,
        indent=2
    )


# =========================================================
# 18. 最终输出
# =========================================================

print("\n" + "=" * 80)
print("本次审核运行完成")
print("=" * 80)

print("模型原始输出：", raw_output_path)
print("规则包：", rule_package_path)
print("完整运行记录：", record_path)


libgomp: Invalid value for environment variable OMP_NUM_THREADS
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

模型和 tokenizer 加载完成
模型加载方式：8-bit
计算类型：float16
模型设备： cuda:0

档案读取完成
有效段落数： 48
正文字符数： 3394

法规解析完成
《档案法》条数： 53
《档案法》已解析条号： [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53]
《实施条例》条数： 52
《实施条例》已解析条号： [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52]
《开放办法》条数： 34
《开放办法》已解析条号： [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34]
规则包完整性检查通过

本次实际调用规则数量： 15

模型输入构造完成
输入 token 数： 5399

Qwen2.5-7B-Instruct 档案开放审核辅助意见
一、事实识别理由

事实1
- 事实内容：档案形成时间为1992年1月16日，已满25年。
- 档案依据：[D005]、[D009]
- 与开放审核的关系：档案已达到法定开放年限。
- 判断：明确适用

事实2
- 事实内容：档案内容涉及风景区规划、建设、环境保护等方面。
- 档案依据：[D003]-[D048]
- 与开放审核的关系：档案内容涉及基础设施建设、道路规划、建筑规划、环境保护等。
- 判断：未发现具体限